# Useful Functions

In [1]:
import csv
from kan import *
from bayes_opt import BayesianOptimization
from sklearn.model_selection import KFold

torch.set_default_dtype(torch.float64)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [2]:
def load_dataset(csv_file_path_list, device='cpu'):

    inputs = []
    labels = []

    for csv_file_path in csv_file_path_list:
        with open(csv_file_path, 'r', newline='', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            for row in reader:
                n = len(row)-1
                input_row = list(map(float, row[0:n]))
                label = int(row[n])
                inputs.append(input_row)
                labels.append(label)

    dataset = {
        'input': torch.tensor(inputs, dtype=torch.double).to(device),
        'label': torch.tensor(labels, dtype=torch.double).to(device)
    }
    # if type is not float silu (in model.fit()) will not run: "silu_cuda" not implemented for 'double'

    return dataset

def standardize_dataset(dataset, mean=None, std=None):
    '''
    dataset is a dictionary with 'input' and 'label' keys, structured as:
        dataset = {
            'input': dataset['input'],
            'label': dataset['label']
        }
    mean and std are statistics with which we standardize the dataset.

    If mean and std are not provided as arguments, then they will be calculated from the dataset.
    '''

    if mean == None:
        mean = dataset['input'].mean(dim=0)
    if std == None:
        std = dataset['input'].std(dim=0)
    standardized = (dataset['input'] - mean) / std
    dataset['input'] = standardized

    return dataset, mean, std

def get_class_weights(dataset):
    '''
    dataset is a dictionary with 'input' and 'label' keys, structured as:
        dataset = {
            'input': dataset['input'],
            'label': dataset['label']
        }
    '''

    # Flatten the labels to 1D for proper bincount calculation
    train_labels_flattened = dataset['label'].int().view(-1).cpu().numpy()

    # Calculate class weights for CrossEntropyLoss based on the class distribution
    class_counts = np.bincount(train_labels_flattened)  # Count occurrences of each class
    num_total = train_labels_flattened.size
    class_weights = num_total / (7 * class_counts)  # Inverse frequency of each class
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    
    return class_weights

def train_model(model, train_loader, opt, loss_fn, steps, lamb=0):

    accuracy_history = []
    loss_history = []
    for i in range(steps):
        train_batch_num = 0
        running_train_accuracy = 0
        running_train_loss = 0

        for batch_inputs, batch_labels in train_loader:
            model.train()

            inputs = batch_inputs.to(device)
            labels = batch_labels.to(device)

            global loss, outputs
            
            # Most optimizers just use optimizer.step(), but certains optimizers require optimizer.step(closure) (e.g., LBFGS). It is used when the optimizer evaluates model and gradient multiple times per step.
            def closure():
                global loss, outputs
                opt.zero_grad()
                outputs = model(inputs)
                # print(outputs.float(), labels.long())
                loss = loss_fn(outputs.float(), labels.long())
                reg = model.get_reg(reg_metric='edge_forward_spline_n', lamb_l1=1., lamb_entropy=1., lamb_coef=0., lamb_coefdiff=0.)
                loss = loss + lamb*reg
                loss.backward()
                return loss
            
            opt.step(closure)
            
            with torch.no_grad():
                preds = torch.argmax(outputs, dim=1)
                batch_accuracy = (torch.sum(preds == labels) / torch.numel(labels)).item()
                running_train_accuracy += batch_accuracy
                batch_loss = loss.item()
                running_train_loss += batch_loss
                train_batch_num += 1

        train_accuracy = running_train_accuracy / train_batch_num
        train_loss = running_train_loss / train_batch_num

        if i % 1 == 0:
            print(f"Epoch: {i}, Train Acc: {train_accuracy}, Loss: {train_loss}")

        accuracy_history.append(train_accuracy)
        loss_history.append(train_loss)
    
    # model.eval()
    # with torch.no_grad():
    #     _ = model(inputs, singularity_avoiding=True)    # refresh safe activations for plotting
    
    return accuracy_history, loss_history

def compute_confusion_matrix(preds, labels, device=None):

    preds = torch.as_tensor(preds)
    labels = torch.as_tensor(labels)

    if device is None:
        device = preds.device

    num_classes = int(max(preds.max(), labels.max()).item() + 1)
    cm = torch.zeros(num_classes, num_classes, dtype=torch.int64, device=device)

    for t, p in zip(labels, preds):
        cm[t, p] += 1

    return cm


def plot_confusion_matrix(cm, class_names=None, normalize=False, title=""):

    if isinstance(cm, torch.Tensor):
        cm = cm.cpu().numpy()

    num_classes = cm.shape[0]
    if class_names is None:
        class_names = [str(i) for i in range(num_classes)]

    if normalize:
        cm = cm.astype(np.float64)
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1  # avoid division by zero
        cm = cm / row_sums

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f" if normalize else "d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(title + (" (Normalized)" if normalize else ""))
    plt.tight_layout()
    plt.show()

def compute_metrics(cm):

    cm = cm.float()
    
    TP = cm.diag()
    precision = TP / (cm.sum(0) + 1e-8)  # sum over column = predicted positives
    recall = TP / (cm.sum(1) + 1e-8)     # sum over row = actual positives
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    
    metrics = {
        'precision_per_class': precision.tolist(),
        'recall_per_class': recall.tolist(),
        'f1_per_class': f1.tolist()
    }
    
    return metrics

def evaluate(model, dataset):
    
    model.eval()

    pred = model(dataset['input']) 
    pred_labels = torch.argmax(pred, dim=1).squeeze()
    true_labels = dataset['label'].int()
    accuracy = (torch.sum(pred_labels == true_labels) / torch.numel(true_labels)).item()

    cm = compute_confusion_matrix(pred_labels, true_labels).cpu()

    return accuracy, cm

# Hyperparameter Optimization

In [ ]:
## Load dataset and split into folds
dataset = load_dataset([r"../data/merged1_train.csv"], device)

print(f"Dataset of size {dataset['input'].shape}")
print(f"Labels of size {dataset['label'].shape}")

dataset = TensorDataset(dataset['input'], dataset['label'])

k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True)

fold_loaders = []

for train_idx, val_idx in kf.split(dataset['input']):

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=len(train_subset), shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=len(val_subset))

    fold_loaders.append((train_loader, val_loader))

fold_loaders_standardized = []

for fold, (train_loader, val_loader) in enumerate(fold_loaders):

    X_train, y_train = next(iter(train_loader)) # This is just a shorthand for “get the first (and only) batch” which contains the entire training fold.
    train_dict = {'input': X_train, 'label': y_train}

    # Standardize training set and compute mean/std
    train_dict, mean, std = standardize_dataset(train_dict)

    class_weights = get_class_weights(train_dict)

    X_val, y_val = next(iter(val_loader))
    val_dict = {'input': X_val, 'label': y_val}

    # Standardize validation set using the corresponding TRAIN mean/std
    val_dict, _, _ = standardize_dataset(val_dict, mean=mean, std=std)

    train_fold = TensorDataset(train_dict['input'], train_dict['label'])
    val_fold = TensorDataset(val_dict['input'], val_dict['label'])

    train_loader_fold = DataLoader(train_fold, batch_size=len(train_fold), shuffle=True)
    val_loader_fold = DataLoader(val_fold, batch_size=len(val_fold), shuffle=False)

    fold_loaders_standardized.append(
        (train_loader_fold, val_loader_fold, class_weights)
    )

In [ ]:
# Train network

def train_model_val(model, train_loader, val_loader, opt, loss_fn, steps, lamb=0):

    # accuracy_history = []
    # loss_history = []
    val_acc_history = []
    # val_loss_history = []

    for i in range(steps):
        train_batch_num = 0
        val_batch_num = 0
        running_train_accuracy = 0
        running_train_loss = 0
        running_val_accuracy = 0
        # running_val_loss = 0

        for batch_inputs, batch_labels in train_loader:
            model.train()

            inputs = batch_inputs.to(device)
            labels = batch_labels.to(device)
            global loss, outputs
            # Most optimizers just use optimizer.step(), but certains optimizers require optimizer.step(closure) (e.g., LBFGS). It is used when the optimizer evaluates model and gradient multiple times per step.
            def closure():
                global loss, outputs
                opt.zero_grad()
                outputs = model(inputs)
                # print(outputs.float(), labels.long())
                loss = loss_fn(outputs.float(), labels.long())
                reg = model.get_reg(reg_metric='edge_forward_spline_n', lamb_l1=1., lamb_entropy=1., lamb_coef=0., lamb_coefdiff=0.)
                loss = loss + lamb*reg
                loss.backward()
                return loss
            
            opt.step(closure)
            with torch.no_grad():
                preds = torch.argmax(outputs, dim=1)
                batch_accuracy = (torch.sum(preds == labels) / torch.numel(labels)).item()
                running_train_accuracy += batch_accuracy
                batch_loss = loss.item()
                running_train_loss += batch_loss
                train_batch_num += 1

        for batch_inputs, batch_labels in val_loader:
            with torch.no_grad():
                inputs = batch_inputs.to(device)
                labels = batch_labels.to(device)
                
                model.eval()
                outputs = model(inputs)

                preds = torch.argmax(outputs, dim=1)

                batch_accuracy = (torch.sum(preds == labels) / torch.numel(labels)).item()
                running_val_accuracy += batch_accuracy

                # loss = loss_fn(outputs.float(), labels.long())
                # reg = model.get_reg(reg_metric='edge_forward_spline_n', lamb_l1=1., lamb_entropy=2., lamb_coef=0., lamb_coefdiff=0.)
                # loss = loss + lamb*reg
                # batch_loss = loss.item()
                # running_val_loss += batch_loss

                val_batch_num += 1

        train_accuracy = running_train_accuracy / train_batch_num
        train_loss = running_train_loss / train_batch_num
        val_accuracy = running_val_accuracy / val_batch_num
        # val_loss = running_val_loss / val_batch_num

        if i % 10 == 0:
            print(f"Epoch: {i}, Train Acc: {train_accuracy}, Loss: {train_loss}, Val Acc: {val_accuracy}")

        # accuracy_history.append(train_accuracy)
        # loss_history.append(train_loss)
        val_acc_history.append(val_accuracy)
        # val_loss_history.append(val_loss)
    
    return val_acc_history


def evaluate_model(log_lr, W, G, log_lamb):
    lr = float(10**log_lr)
    W = int(round(W))
    G = int(round(G))
    lamb = float(10**log_lamb)
    #K = int(K)

    fold_accuracies = []

    for fold, (train_loader, val_loader, class_weights) in enumerate(fold_loaders_standardized):
        print(f"Training fold {fold}")

        model = KAN(width=[12, W, 5], grid=G, k=3, device=device)

        if torch.cuda.device_count() > 1:
            print("Using", torch.cuda.device_count(), "GPUs!")
            model = nn.DataParallel(model)

        optimizer = LBFGS(model.parameters(), lr)
        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

        val_acc_history = train_model_val(model, train_loader, val_loader, opt=optimizer, loss_fn=criterion, steps=100, lamb=lamb)
        acc = float(val_acc_history[-1])
        fold_accuracies.append(acc)
    # model.plot()

    mean_accuarcy = sum(fold_accuracies) / len(fold_accuracies)

    print("Fold accs:", fold_accuracies)
    print("Mean acc:", mean_accuarcy)

    return mean_accuarcy

# Define search space
pbounds = {
    'log_lr': (-4, -1),     # sweeps log_lr uniformly -> sweeps 10**log_lr on log scale
    'W': (3, 10),
    'G': (3, 10),
    'log_lamb': (-4, -1)      
}

# Run Bayesian Optimization
optimizer = BayesianOptimization(
    f=evaluate_model,
    pbounds=pbounds,
    verbose=2
)

optimizer.maximize(
    init_points=8,  # random start
    n_iter=32       # number of guided iterations
)

print("Best result:")
print(optimizer.max)

# Train Model

In [ ]:
## Select model

# Create a KAN: width=(num_inputs, num_hidden_neurons, num_outputs), spline order (k), grid intervals (grid).
model = KAN(width=[12,9,5], grid=8, k=3, =42, device=device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total number of parameters: {total_params}')

In [ ]:
# Load the train dataset
dataset = load_dataset([r"../data/merged1_train.csv"], device)
dataset, train_mean, train_std = standardize_dataset(dataset)
class_weights = get_class_weights(dataset)

print(f"Dataset of size {dataset['input'].shape}")
print(f"Labels of size {dataset['label'].shape}")

print(f"Mean of dataset is {train_mean}")
print(f"Standard deviation of dataset is {train_std}")

print(f"Class weights are {class_weights}")

train_loader = DataLoader(TensorDataset(dataset['input'], dataset['label']), batch_size=dataset['input'].shape[0])

# Load the test dataset
test_dataset = load_dataset([r"../data/merged1_test.csv"], device)
test_dataset, _, _ = standardize_dataset(test_dataset, mean=train_mean, std=train_std)
noise = torch.normal(mean=0, std=0, size=test_dataset['input'].shape).to(device)
test_dataset['input'] = test_dataset['input'] + noise

print(f"Dataset of size {test_dataset['input'].shape}")
print(f"Labels of size {test_dataset['label'].shape}")

In [ ]:
## Train network: Initial training, pruning, and symbolic formula extraction
## Test after each stage and print metrics

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model)

optimizer = LBFGS(model.parameters(), lr=1e-1)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

## Train initial model
print("Training initial model")
accuracy_history, loss_history = train_model(model, train_loader, opt=optimizer, loss_fn=criterion, steps=100, lamb=1.3e-4)
# model.refine(model.grid*2)
# accuracy_history, loss_history, val_acc_history, val_loss_history = train_model(model, train_loader, val_loader, opt=optimizer, loss_fn=criterion, steps=100, lamb=0)
model.log_history('fit')    # save model
model.plot()

# Print loss curve
plt.figure()
plt.plot(loss_history)
plt.legend(['train'])
plt.ylabel('loss')
plt.xlabel('step')
plt.show()

# Print training curve
plt.figure()
plt.plot(accuracy_history)
plt.legend(['train'])
plt.ylabel('accuracy')
plt.xlabel('step')
plt.show()

# Test initial model
accuracy, cm = evaluate(model, test_dataset)
plot_confusion_matrix(cm, class_names=['benign', 'DoS', 'crypto', 'backdoor', 'recon'], normalize=False, title="")
print(f"Test Accuracy: {accuracy}")

metrics = compute_metrics(cm)
print("Precision per class:", metrics['precision_per_class'])
print("Recall per class:", metrics['recall_per_class'])
print("F1 per class:", metrics['f1_per_class'])

# total_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {total_params}")

## Prune model
print("Pruning model")
model = model.prune()
model.plot()

# Re-train
optimizer = LBFGS(model.parameters(), lr=1e-1)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
accuracy_history, loss_history = train_model(model, train_loader, opt=optimizer, loss_fn=criterion, steps=100, lamb=1.3e-4)
model.log_history('fit')    # save model
model.plot()

# Print loss curve
plt.figure()
plt.plot(loss_history)
plt.legend(['train'])
plt.ylabel('loss')
plt.xlabel('step')
plt.show()

# Print training curve
plt.figure()
plt.plot(accuracy_history)
plt.legend(['train'])
plt.ylabel('accuracy')
plt.xlabel('step')
plt.show()

# Test model
accuracy, cm = evaluate(model, test_dataset)
plot_confusion_matrix(cm, class_names=['benign', 'DoS', 'crypto', 'backdoor', 'recon'], normalize=False, title="")
print(f"Test Accuracy: {accuracy}")

metrics = compute_metrics(cm)
print("Precision per class:", metrics['precision_per_class'])
print("Recall per class:", metrics['recall_per_class'])
print("F1 per class:", metrics['f1_per_class'])

# total_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {total_params}")

## Extract symbolic formula
print("Extracting symbolic formula")
mode = "auto" # "manual"
if mode == "manual":
    # manual mode
    model.fix_symbolic(0,0,0,'sin')
    model.fix_symbolic(0,1,0,'x^2')
    model.fix_symbolic(1,0,0,'exp')
elif mode == "auto":
    # automatic mode
    lib = ['x','x^2','x^3','x^4','exp','log','sqrt','tanh','sin','abs']
    model.auto_symbolic(lib=lib, weight_simple=0.8)

# Re-train
optimizer = LBFGS(model.parameters(), lr=1e-1)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
accuracy_history, loss_history = train_model(model, train_loader, opt=optimizer, loss_fn=criterion, steps=100, lamb=0)
model.log_history('fit')    # save model
model.plot()

# Test model
accuracy, cm = evaluate(model, test_dataset)
plot_confusion_matrix(cm, class_names=['benign', 'DoS', 'crypto', 'backdoor', 'recon'], normalize=False, title="")
print(f"Test Accuracy: {accuracy}")

metrics = compute_metrics(cm)
print("Precision per class:", metrics['precision_per_class'])
print("Recall per class:", metrics['recall_per_class'])
print("F1 per class:", metrics['f1_per_class'])

# Print symbolic formulas
from kan.utils import ex_round

for formula in model.symbolic_formula()[0]:
    print(ex_round(formula, 4))
    print('')


# Testing

In [ ]:
# Load the training data (features)
dataset = load_dataset([r"../data/merged1_train.csv"], device)
dataset, train_mean, train_std = standardize_dataset(dataset)
class_weights = get_class_weights(dataset)

print(f"Dataset of size {dataset['input'].shape}")
print(f"Labels of size {dataset['label'].shape}")

print(f"Mean of dataset is {train_mean}")
print(f"Standard deviation of dataset is {train_std}")

print(f"Class weights are {class_weights}")

train_loader = DataLoader(TensorDataset(dataset['input'], dataset['label']), batch_size=dataset['input'].shape[0])

# Load the test dataset (features)
test_dataset = load_dataset([r"../data/merged1_test.csv"], device)
test_dataset, _, _ = standardize_dataset(test_dataset, mean=train_mean, std=train_std)
noise = torch.normal(mean=0, std=0, size=test_dataset['input'].shape).to(device)
test_dataset['input'] = test_dataset['input'] + noise

print(f"Dataset of size {test_dataset['input'].shape}")
print(f"Labels of size {test_dataset['label'].shape}")


In [ ]:
# Decision Tree

import pickle, sys

X = dataset['input'].cpu().numpy()
y = dataset['label'].cpu().numpy().astype(int)

X_test = test_dataset['input'].cpu().numpy()
y_test = test_dataset['label'].cpu().numpy().astype(int)

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

def train_tree(X_train, y_train, class_weights):

    model = DecisionTreeClassifier(
        class_weight=class_weights,
        max_depth=8,
    )

    model.fit(X_train, y_train)

    return model

class_weight_dict = {i: class_weights[i] for i in range(0, len(class_weights))}

model = train_tree(X, y, class_weight_dict)

# Save tree to filename
filename = 'decision_tree_model.pickle'
with open(filename, 'wb') as file:
    pickle.dump(model, file)

preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {acc}")
# print(f"Node count: {model.tree_.node_count}")

cm = compute_confusion_matrix(preds, y_test).cpu()
metrics = compute_metrics(cm)
print("Precision per class:", metrics['precision_per_class'])
print("Recall per class:", metrics['recall_per_class'])
print("F1 per class:", metrics['f1_per_class'])

total_params = sum(arr.size for arr in [
    model.tree_.feature,
    model.tree_.threshold,
    model.tree_.children_left,
    model.tree_.children_right,
    model.tree_.value
])

print(f"Total parameters: {total_params}")

# with open(filename, 'rb') as file:
#     loaded_model = pickle.load(file)
# tree_size = sys.getsizeof(pickle.dumps(model))
# print(f"Decision Tree Size: {tree_size} bytes")

# # Plot
# plt.figure(figsize=(12,8))
# plot_tree(
#     model,
#     filled=True,
#     rounded=True,
#     fontsize=10
# )
# plt.savefig("decision_tree.png", dpi=300, bbox_inches='tight')
# plt.close()

In [ ]:
## MLP

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, device="cuda"):
        super(MLP, self).__init__()

        self.device = device
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input):
        x = self.dropout(F.relu(self.fc1(input)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)
        return x
    
model = MLP(input_dim=12, hidden_dim=64, output_dim=5, device=device).to(device)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights.double())

X_train = dataset['input'].to(device)
y_train = dataset['label'].long().to(device)

best_accuracy = 0.0
best_model_state = None

for epoch in range(1500):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train)
    loss = criterion(preds, y_train)
    loss.backward()
    optimizer.step()
    predicted = torch.argmax(preds, dim=1)
    accuracy = (predicted == y_train).float().mean().item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Accuracy: {accuracy} Loss: {loss.item():.4f}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model_state = model.state_dict()

model.load_state_dict(best_model_state)
filename = 'MLP.pth'
torch.save(model.state_dict(), filename)

# model_state_dict = torch.load("MLP.pth", weights_only=True)
# model.load_state_dict(model_state_dict)
model.eval()

X_test = test_dataset['input']
y_test = test_dataset['label']

with torch.no_grad():
    preds = model(X_test)
    predicted = torch.argmax(preds, dim=1)

accuracy = (predicted == y_test).float().mean().item()

print("Test Accuracy:", accuracy)
cm = compute_confusion_matrix(predicted.long(), y_test.long()).cpu()
metrics = compute_metrics(cm)
print("Precision per class:", metrics['precision_per_class'])
print("Recall per class:", metrics['recall_per_class'])
print("F1 per class:", metrics['f1_per_class'])

# total_params = sum(p.numel() for p in model.parameters())
# print(f'Total number of parameters: {total_params}')
# total_train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f'Total number of trainable parameters: {total_train_params}')

In [ ]:
# Load the training data (time series)
dataset = load_dataset([r"../data/merged2_train.csv"], device)

dataset, train_mean, train_std = standardize_dataset(dataset)
class_weights = get_class_weights(dataset)

print(f"Dataset of size {dataset['input'].shape}")
print(f"Labels of size {dataset['label'].shape}")

print(f"Mean of dataset is {train_mean}")
print(f"Standard deviation of dataset is {train_std}")

print(f"Class weights are {class_weights}")

train_loader = DataLoader(TensorDataset(dataset['input'], dataset['label']), batch_size=dataset['input'].shape[0])

# Load the test dataset
test_dataset = load_dataset([r"../data/merged2_test.csv"], device)
test_dataset, _, _ = standardize_dataset(test_dataset, mean=train_mean, std=train_std)
noise = torch.normal(mean=0, std=0, size=test_dataset['input'].shape).to(device)
test_dataset['input'] = test_dataset['input'] + noise

print(f"Dataset of size {test_dataset['input'].shape}")
print(f"Labels of size {test_dataset['label'].shape}")

In [ ]:
# CNN

class SimpleCNN(nn.Module):
    def __init__(self, output_dim=5):
        super(SimpleCNN, self).__init__()

        # convolution layers
        self.conv1 = nn.Conv1d(4, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(16, 16, kernel_size=3, padding=1)

        self.pool = nn.MaxPool1d(2)

        # fully connected
        self.fc1 = nn.Linear(16 * 5, 32)
        self.fc2 = nn.Linear(32, output_dim)

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)

        return x
    
model = SimpleCNN(output_dim=5).to(device)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device).double())

X_train = dataset['input'].view(-1, 20, 4)      # (batch, time, features)
X_train = X_train.permute(0, 2, 1).to(device)     # (batch, channels, sequence)
y_train = dataset['label'].long().to(device)

best_accuracy = 0.0
best_model_state = None

for epoch in range(1500):
    model.train()
    optimizer.zero_grad()

    preds = model(X_train)
    loss = criterion(preds, y_train)

    loss.backward()
    optimizer.step()

    predicted = torch.argmax(preds, dim=1)
    accuracy = (predicted == y_train).float().mean().item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Accuracy: {accuracy:.4f}, Loss: {loss.item():.4f}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model_state = model.state_dict()

model.load_state_dict(best_model_state)

filename = "CNN.pth"
torch.save(model.state_dict(), filename)

# model_state_dict = torch.load("CNN.pth", weights_only=True)
# model.load_state_dict(model_state_dict)
model.eval()

X_test = test_dataset['input'].view(-1, 20, 4)      # (batch, time, features)
X_test = X_test.permute(0, 2, 1).to(device)     # (batch, channels, sequence)
y_test = test_dataset['label'].long().to(device)

with torch.no_grad():
    preds = model(X_test)
    predicted = torch.argmax(preds, dim=1)

accuracy = (predicted == y_test).float().mean().item()

print("Test Accuracy:", accuracy)

cm = compute_confusion_matrix(predicted.long(), y_test.long()).cpu()
metrics = compute_metrics(cm)
print("Precision per class:", metrics['precision_per_class'])
print("Recall per class:", metrics['recall_per_class'])
print("F1 per class:", metrics['f1_per_class'])

# total_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {total_params}")

# total_train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Trainable parameters: {total_train_params}")


In [ ]:
# LSTM
class SimpleLSTM(nn.Module):
    def __init__(self, input_size=4, hidden_size=32, num_layers=1, output_dim=5, dropout=0.1):
        super(SimpleLSTM, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq_len, features)
        out, (hn, cn) = self.lstm(x)

        # Take last time step
        out = out[:, -1, :]  # (batch, hidden_size)

        out = self.dropout(F.relu(out))
        out = self.fc(out)

        return out

# (batch, seq_len, features)
X_train = dataset['input'].view(-1, 20, 4).to(device)
y_train = dataset['label'].long().to(device)

X_test = test_dataset['input'].view(-1, 20, 4).to(device)
y_test = test_dataset['label'].long().to(device)

model = SimpleLSTM(input_size=4, hidden_size=32, num_layers=1, output_dim=5, dropout=0.1).to(device)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device).double())

best_accuracy = 0.0
best_model_state = None

for epoch in range(1000):
    model.train()
    optimizer.zero_grad()
    
    preds = model(X_train)
    
    loss = criterion(preds, y_train)
    
    loss.backward()
    optimizer.step()
    
    predicted = torch.argmax(preds, dim=1)
    accuracy = (predicted == y_train).float().mean().item()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Accuracy: {accuracy:.4f}, Loss: {loss.item():.4f}")
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model_state = model.state_dict()

model.load_state_dict(best_model_state)

filename = "LSTM.pth"
torch.save(model.state_dict(), filename)

# model_state_dict = torch.load("LSTM.pth", weights_only=True)
# model.load_state_dict(model_state_dict)

model.eval()

with torch.no_grad():
    preds = model(X_test)
    predicted = torch.argmax(preds, dim=1)

accuracy = (predicted == y_test).float().mean().item()
print("Test Accuracy:", accuracy)

cm = compute_confusion_matrix(predicted.long(), y_test.long()).cpu()
metrics = compute_metrics(cm)
print("Precision per class:", metrics['precision_per_class'])
print("Recall per class:", metrics['recall_per_class'])
print("F1 per class:", metrics['f1_per_class'])

# total_params = sum(p.numel() for p in model.parameters())
# total_train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Total parameters: {total_params}")
# print(f"Trainable parameters: {total_train_params}")

In [ ]:
# MiniRocket
from tsai.models.MINIROCKET_Pytorch import MiniRocketFeatures

class MiniRocketNet(nn.Module):
    def __init__(self, input_channels=4, seq_len=20, num_classes=5):
        super().__init__()

        # MiniRocket feature extractor
        self.mr = MiniRocketFeatures(
            c_in=input_channels,
            seq_len=seq_len,
            num_features=1000
        )

        # small classifier head
        self.fc = nn.Linear(924, num_classes)

    def forward(self, x):
        # x expected shape: (batch, channels, seq_len)
        feats = self.mr(x)        # -> (batch, num_features)
        out = self.fc(feats)      # -> (batch, num_classes)
        return out
    
# (batch, channels, sequence_length)
X_train = dataset['input'].view(-1, 20, 4)      # (batch, time, features)
X_train = X_train.permute(0, 2, 1).float().to(device)     # (batch, channels, sequence)
y_train = dataset['label'].long().to(device)

X_test = test_dataset['input'].view(-1, 20, 4)      # (batch, time, features)
X_test = X_test.permute(0, 2, 1).float().to(device)     # (batch, channels, sequence)
y_test = test_dataset['label'].long().to(device)

model = MiniRocketNet(input_channels=4, seq_len=20, num_classes=5).to(device)

model.float()

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device).float())

best_accuracy = 0.0
best_model_state = None

for epoch in range(1200):
    model.train()   
    optimizer.zero_grad()

    preds = model(X_train)
    loss = criterion(preds, y_train)

    loss.backward()
    optimizer.step()

    accuracy = (preds.argmax(1) == y_train).float().mean().item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Accuracy: {accuracy:.4f}, Loss: {loss.item():.4f}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model_state = model.state_dict()

model.load_state_dict(best_model_state)

filename = "MiniRocket.pth"
torch.save(model.state_dict(), filename)

# model_state_dict = torch.load("MiniRocket.pth", weights_only=True)
# model.load_state_dict(model_state_dict)

model.eval()

with torch.no_grad():
    preds = model(X_test)
    test_acc = (preds.argmax(1) == y_test).float().mean().item()

print("Test Accuracy:", test_acc)

cm = compute_confusion_matrix(predicted.long(), y_test.long()).cpu()
metrics = compute_metrics(cm)
print("Precision per class:", metrics['precision_per_class'])
print("Recall per class:", metrics['recall_per_class'])
print("F1 per class:", metrics['f1_per_class'])

# total_params = sum(p.numel() for p in model.parameters())
# total_train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Total parameters: {total_params}")
# print(f"Trainable parameters: {total_train_params}")

In [ ]:
def symbolic_model(x):
    """
    Symbolic regression model with 5 outputs.
    """

    x = np.atleast_2d(x)  # shape (N, 12)

    x_1  = x[:, 0]
    x_2  = x[:, 1]
    x_3  = x[:, 2]
    x_4  = x[:, 3]
    x_5  = x[:, 4]
    x_6  = x[:, 5]
    x_7  = x[:, 6]
    x_8  = x[:, 7]
    x_9  = x[:, 8]
    x_10 = x[:, 9]
    x_11 = x[:, 10]
    x_12 = x[:, 11]

    y1 = (33.122*x_1 - 4.5041*x_10 + 0.8549*x_11 + 0.4755*x_12
          - 1.935*x_2 - 0.6377*x_3 + 16.5232*x_4 - 0.865*x_5
          + 0.2456*x_6 - 2.3685*x_7 - 0.0631*x_8 + 0.3321*x_9
          + 43.0954*np.sin(1.2236*x_1 + 7.8663)
          - 0.0455*np.sin(2.0105*x_1 + 3.0724)
          - 30.3822)

    y2 = (-35.3947*x_1 - 2.1703*x_10 + 6.5943*x_11 - 0.5471*x_12
          + 1.667*x_2 + 0.7337*x_3 - 8.3336*x_4 + 1.129*x_5
          - 0.3124*x_6 - 0.6443*x_7 + 2.6935*x_8 - 0.2565*x_9
          - 33.3229*np.sin(1.2236*x_1 + 7.8663)
          - 18.1992*np.sin(-3.0471*x_1 + 1.2497*x_10 + 0.2733*x_11
                           + 0.212*x_3 + 0.3862*x_4 + 0.446*x_7 - 7.5014)
          + 14.6336)

    y3 = (9.2667*x_1 - 13.2377*x_10 + 3.1412*x_11 + 0.3788*x_12
          + 0.1392*x_2 - 0.508*x_3 + 38.5163*x_4 + 0.6002*x_5
          + 2.1094*x_6 + 4.1434*x_7 + 1.5323*x_8 + 0.0261*x_9
          - 17.9223*np.sin(1.2236*x_1 + 7.8663)
          - 16.4684*np.sin(2.0105*x_1 + 3.0724)
          + 27.869*np.sin(0.5196*x_10 - 7.7258)
          + 41.299)

    y4 = (-3.6728*x_1 - 4.9731*x_10 + 2.3779*x_11 - 0.0763*x_12
          - 0.4579*x_2 - 0.3051*x_3 + 14.164*x_4 + 0.3226*x_5
          + 2.0852*x_6 - 2.5659*x_7 + 1.7128*x_8 + 0.1567*x_9
          - 15.737*np.sin(2.0105*x_1 + 3.0724)
          + 8.0449)

    y5 = (-8.9576*x_1 - 22.2662*x_10 + 2.9281*x_11 - 0.8854*x_12
          + 1.9994*x_2 - 2.9711*x_3 - 5.8558*x_4 + 0.1719*x_6
          + 0.6118*x_7 + 0.159*x_8 + 0.0227*x_9
          + 1.3416*np.sin(1.2236*x_1 + 7.8663)
          - 1.244*np.sin(2.0105*x_1 + 3.0724)
          - 3.224)

    return [y1, y2, y3, y4, y5]

In [ ]:
# Measure inference time
import time

def measure_inference_time(model, input_tensor, warmup_runs=10):

    # Force CPU usage with single thread
    device = torch.device('cpu')
    torch.set_num_threads(1)
    model = model.to(device)
    model.eval()
    input_tensor = input_tensor.to(device)

    # Warmup runs
    with torch.no_grad():
        for i in range(warmup_runs):
            _ = model(input_tensor[i].unsqueeze(0))

    # Timed runs
    latencies = []
    with torch.no_grad():
        for input in input_tensor:
            start = time.perf_counter()
            _ = model(input.unsqueeze(0))
            end = time.perf_counter()
            latencies.append((end - start) * 1000)  # Convert to ms

    latencies = np.array(latencies)

    mean_ms = np.mean(latencies)
    std_ms = np.std(latencies)

    return mean_ms, std_ms

def measure_dt_inference_time(model, input_tensor, warmup_runs=10):

    # Force CPU
    device = torch.device('cpu')
    torch.set_num_threads(1)
    input_tensor = input_tensor.to(device)

    # Warmup runs
    with torch.no_grad():
        for i in range(warmup_runs):
            _ = model.predict(input_tensor[i].unsqueeze(0))

    # Timed runs
    latencies = []
    with torch.no_grad():
        for input in input_tensor:
            start = time.perf_counter()
            _ = model.predict(input.unsqueeze(0))
            end = time.perf_counter()
            latencies.append((end - start) * 1000)  # Convert to ms
            
    latencies = np.array(latencies)

    mean_ms = np.mean(latencies)
    std_ms = np.std(latencies)

    return mean_ms, std_ms

# Replace with model and input
# model = KAN.loadckpt('./model/' + '0.5')
model = SimpleCNN(output_dim=5)
model_state_dict = torch.load("CNN.pth", weights_only=True)
model.load_state_dict(model_state_dict)

# print(X_test.shape)   # ensure dataset has desired shape

mean_ms, std_ms = measure_inference_time(model, X_test, warmup_runs=10)

# with open('decision_tree_model.pickle', 'rb') as file:
#     model = pickle.load(file)
# print(test_dataset["input"].shape)
# mean_ms, std_ms = measure_dt_inference_time(model, test_dataset["input"], warmup_runs=10)

print(f"mean_ms: {mean_ms} ms")
print(f"std_ms: {std_ms} ms")